In [ ]:
import os
os.environ['AWS_PROFILE'] = 'admin'
os.environ['HAVEN_DATABASE'] = 'haven'

import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
import numpy as np
import h3
from tqdm import tqdm

from mirrorverse.utils import read_data_w_cache
from mirrorverse.plotting import build_geojson

from sklearn.decomposition import PCA 

In [ ]:
sql = '''
select 
    h3_index,
    chlorophyll, 
    time,
    extract(year from time) as year,
    extract(month from time) as month
from 
    copernicus_biochemistry
where 
    h3_resolution = 4
    and depth_bin = 25.0
    and extract(year from time) in (2015, 2016)
    and extract(day from time) = 1
'''
raw_data = read_data_w_cache(sql)
raw_data['lat'] = raw_data['h3_index'].apply(lambda h: h3.h3_to_geo(h)[0])
raw_data['lon'] = raw_data['h3_index'].apply(lambda h: h3.h3_to_geo(h)[1])
raw_data['epoch'] = raw_data['time'].astype('int64') // 10**9
raw_data['raw_chlorophyll'] = raw_data['chlorophyll']

raw_data = raw_data[(raw_data['lon'] > -170) & (raw_data['lat'] > 42) & (raw_data['lat'] < 64)]
raw_data = raw_data.sort_values(['epoch', 'h3_index'], ascending=False).reset_index(drop=True)
print(raw_data.shape)
raw_data.head()

In [ ]:
px.histogram(raw_data['raw_chlorophyll'])

In [ ]:
raw_data['chlorophyll'] = (raw_data['raw_chlorophyll'] + 1) ** 0.25

In [ ]:
px.histogram(raw_data['chlorophyll'])

In [ ]:
rows = []
for h3_index in tqdm(list(raw_data['h3_index'].unique())):
    df = raw_data[(raw_data['h3_index'] == h3_index)].reset_index(drop=True)
    for i, entry in df.iterrows():
        row = {
            'year': entry['year'],
            'month': entry['month'],
            'h3_index': entry['h3_index'],
        }
        vals = list(df['chlorophyll'].values[i:entry['month']+i])[::-1]
        if len(vals) < entry['month']:
            break
        for j, val in enumerate(vals):
            row[f'chlorophyll_{j}'] = val
        rows.append(row)
data = pd.DataFrame(rows)
print(data.shape)
data.head()

In [ ]:
X = np.array(data[data['month'] == 12][[f'chlorophyll_{i}' for i in range(12)]])
X_mean = np.mean(X, axis=0)
X = X - X_mean
pca = PCA(n_components=12)
pca.fit(X)
pca.explained_variance_ratio_

In [ ]:
dfs = []
for month in tqdm(list(range(1, 13))):
    df = data[data['month'] == month].copy()
    X = np.array(df[[f'chlorophyll_{i}' for i in range(month)]])
    X = X - X_mean[:month]
    for i, component in enumerate(pca.components_):
        component = component[:month]
        df[f'component_{i}'] = np.dot(X, component)
    dfs.append(df)
comps = pd.concat(dfs).sort_values(['h3_index', 'month'], ascending=False).reset_index(drop=True)
print(comps.shape)
comps.head()

In [ ]:
comps['extent'] = comps.apply(
    lambda r: (r['component_0'] ** 2 + r['component_1'] ** 2 + r['component_2'] ** 2) ** 0.5,
    axis=1
)
comps['tendency_0'] = comps['component_0'] / comps['extent']
comps['tendency_1'] = comps['component_1'] / comps['extent']
comps['tendency_2'] = comps['component_2'] / comps['extent']
comps.head()

In [ ]:
df = comps[comps['year'] == 2016].copy()
df['lat'] = df['h3_index'].apply(lambda x: h3.h3_to_geo(x)[0])
df['lon'] = df['h3_index'].apply(lambda x: h3.h3_to_geo(x)[1])
df = df[(df['lon'] > -170) & (df['lon'] < 0) & (df['lat'] > 42) & (df['lat'] < 64)]
feature = 'component_1'

fig = go.Figure()
geojson = build_geojson(df, 'h3_index')
months = sorted(df['month'].unique())
for month in months:
    sdf = df[df['month'] == month]
    fig.add_trace(
        go.Choroplethmapbox(
            geojson=geojson,
            locations=sdf['h3_index'],
            z=sdf[feature],
            visible=False,
            marker_line_color='rgba(255,255,255,0)',
            zmin=df[feature].min(),
            zmax=df[feature].max(),
            colorscale='algae'
        )
    )

fig.data[0].visible = True

steps = []
for i, slider_val in enumerate(months):
    step = dict(
        method="update",
        args=[
            {"visible": [False] * len(months)},
            {"title": f"month: {slider_val}"},
        ],
        label=f"{slider_val}"
    )
    step["args"][0]["visible"][i] = True
    steps.append(step)

sliders = [dict(
    active=0,
    currentvalue={"prefix": f"feature: "},
    pad={"t": 50, "b": 25, "l": 25},
    steps=steps
)]

fig.update_layout(
    sliders=sliders
)

fig.update_layout(
    autosize=False,  # Disable autosizing
    width=800,       # Set width in pixels
    height=800,      # Set height in pixels
)

fig.update_layout(
    margin={"r":0,"t":30,"l":0,"b":0}, mapbox=dict(style="carto-positron", zoom=3, center = {"lat": 57, "lon": -150})
)

fig.show()

In [ ]:
import plotly.graph_objects as go
import pandas as pd

# Normalize the components to 0-255 for RGB
def normalize(series):
    return ((series - series.min()) / (series.max() - series.min()) * 255).astype(int)

df['r'] = normalize(df['component_0'])
df['g'] = normalize(df['component_1'])
df['b'] = normalize(df['component_2'])
df['color'] = df.apply(lambda row: f'rgb({row.r},{row.g},{row.b})', axis=1)

# Create one scatter per month
months = sorted(df['month'].unique())
fig = go.Figure()

for i, month in enumerate(months):
    subset = df[df['month'] == month]
    fig.add_trace(go.Scattergeo(
        lon=subset['lon'],
        lat=subset['lat'],
        mode='markers',
        marker=dict(
            color=subset['color'],
            size=6,
            opacity=0.7
        ),
        name=f'Month {month}',
        visible=(i == 0)  # Only show first month initially
    ))

# Create slider steps
steps = []
for i, month in enumerate(months):
    step = dict(
        method='update',
        args=[{'visible': [j == i for j in range(len(months))]},
              {'title': f'Month: {month}'}],
        label=str(month)
    )
    steps.append(step)

# Add sliders
sliders = [dict(
    active=0,
    currentvalue={'prefix': 'Month: '},
    pad={"t": 50},
    steps=steps
)]

fig.update_layout(
    title='RGB Scatter Plot by Month',
    geo=dict(
        projection_type='equirectangular',
        showland=True,
        landcolor="rgb(217, 217, 217)",
        showcountries=True,
    ),
    sliders=sliders,
)

fig.show()


In [ ]:
dfs = []
for i in range(3):
    df = pd.DataFrame(pca.components_[i], columns=['value']).reset_index().rename(columns={'index': 'month'})
    df['component'] = str(i)
    dfs.append(df)
df = pd.concat(dfs)
px.line(df, x='month', y='value', color='component')